### Summary

#### MCP Servers
2 mcp servers are used for discovering files in the repo and check the codes in it
- local for discovering files in the project: code-index-mcp (https://glama.ai/mcp/servers/johnhuang316/code-index-mcp)
- api based for analysing files: mcp-code-sanitizer (https://glama.ai/mcp/servers/notasandy/mcp-code-sanitizer)

#### UI
Use Gradio chat to discover your project and analyse the code like:
- "find me Python files which potentially have security issues" 
- "Analyse the code in the first file"

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)

In [ ]:
# Define the project path, if needed. If None, it will use the the community contributions directory as the project path.
PROJECT_PATH = None

project_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if not PROJECT_PATH else os.path.abspath(PROJECT_PATH)
print(f"Project directory: {project_dir}")

# Set the model
model = "gpt-4.1-mini"

# Need to set the project path in the instructions for the agent, so it can use it when needed.
instructions = f"Set the project path to this: {project_dir}. If you return any file paths, make sure they are absolute paths"

In [ ]:
sanitizer_params = [{"command": "uvx","args": ["code-index-mcp"]}, {"command": "uvx","args": ["mcp-code-sanitizer"], "env": {"GROQ_API_KEY": os.getenv("GROQ_API_KEY")}}]
mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in sanitizer_params]

In [ ]:
import gradio as gr

async def chat(message, history):
    conversation = [
        {"role": item["role"], "content": item["content"]}
        for item in history
    ] + [{"role": "user", "content": message}]
    for server in mcp_servers:
        await server.connect()
    agent = Agent(
        name="Code Sanitizer Chat",
        instructions=instructions,
        model=model,
        mcp_servers=mcp_servers,
    )
    with trace("code_sanitizer_chat"):
        result = await Runner.run(agent, conversation)
    return result.final_output

gr.ChatInterface(
    fn=chat,
    type="messages",
    title="Code Sanitizer Chat",
    description="Ask questions about the indexed project. The agent uses code-sanitizer-mcp tools when needed.",
).launch()